# 🎙️ Treinamento da Voz Oficial do Mascote FEFO (F5-TTS Fine-Tuning)
**Projeto FEFO** — Notebook 100% autônomo e auto-recuperável para treinar uma IA dedicada com as gravações oficiais do FEFO (577 frases / 25 minutos de áudio).

---
### ⚡ Instruções (Apenas 1 Clique):
1. Verifique se a GPU está ativa: **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ **T4 GPU** ➔ **Salvar**.
2. Clique em **Ambiente de execução** ➔ **Executar tudo** (ou pressione `Ctrl + F9`).
3. O notebook vai instalar tudo sozinho, preparar os 577 áudios, treinar na GPU (~35-45 min), tocar o teste e baixar o modelo!

In [ ]:
# 1. Verificação da GPU NVIDIA
import torch
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada!')
    print('Vá em: Ambiente de execução -> Alterar tipo de ambiente -> Escolha T4 GPU.')
print('=' * 60)


In [ ]:
# 2. Preparação Completa e Treinamento da Voz do FEFO na GPU T4
import os, sys, shutil, zipfile, glob

# 1. Garantir diretório raiz
os.chdir('/content')

# 2. Baixar repositório do FEFO com o dataset oficial (577 áudios)
if not os.path.exists('/content/fefo-repo'):
    print('[1/5] Clonando repositório oficial do FEFO...')
    !git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo

# 3. Instalar o F5-TTS e dependências
if not os.path.exists('/content/F5-TTS'):
    print('[2/5] Instalando F5-TTS e pacotes de áudio...')
    !git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
    %cd /content/F5-TTS
    !pip install -q -e .
    !pip install -q huggingface_hub vocos soundfile torchaudio accelerate pydub gdown

# 4. Descompactar dataset dos áudios do FEFO
dataset_raw = '/content/F5-TTS/data/fefo'
if not os.path.exists(f'{dataset_raw}/metadata.csv'):
    print('[3/5] Extraindo 577 áudios oficiais do mascote FEFO...')
    os.makedirs(dataset_raw, exist_ok=True)
    zip_path = '/content/fefo-repo/fefo_training_dataset.zip'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(dataset_raw)
    print(f'✓ {len(os.listdir(dataset_raw + "/wavs"))} clipes de áudio prontos!')

# 5. Estruturar o dataset com prepare_csv_wavs.py
dataset_pinyin = '/content/F5-TTS/data/fefo_pinyin'
if not os.path.exists(f'{dataset_pinyin}/raw.arrow'):
    print('[4/5] Estruturando vocabulário fonético e metadados...')
    %cd /content/F5-TTS
    !python src/f5_tts/train/datasets/prepare_csv_wavs.py /content/F5-TTS/data/fefo/metadata.csv /content/F5-TTS/data/fefo_pinyin

# 6. Disparar o treinamento com GPU T4 (20 épocas)
print('=' * 60)
print('[5/5] INICIANDO O TREINAMENTO NA GPU T4 (20 épocas / ~35-45 min)...')
print('A barra de progresso abaixo começará a avançar agora:')
print('=' * 60)
%cd /content/F5-TTS
!accelerate launch src/f5_tts/train/finetune_cli.py \
    --exp_name F5TTS_v1_Base \
    --dataset_name fefo \
    --finetune \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 3200 \
    --batch_size_type frame \
    --epochs 20 \
    --save_per_updates 500


In [ ]:
# 3. Teste de Áudio com a Voz Treinada e Download do Modelo
import os, glob, shutil, IPython.display as ipd
from google.colab import files

%cd /content/F5-TTS

# Localizar melhor checkpoint gerado
ckpts = glob.glob('/content/F5-TTS/ckpts/*/*.pt') + glob.glob('/content/F5-TTS/ckpts/*/*.safetensors')
if ckpts:
    best_ckpt = sorted(ckpts)[-1]
    print(f'✓ Modelo treinado localizado: {best_ckpt}')
    
    # Gerar áudio de teste com o FEFO
    cfg_files = glob.glob('/content/F5-TTS/src/f5_tts/configs/*.yaml')
    model_name = 'F5TTS_v1_Base' if any('F5TTS_v1_Base' in c for c in cfg_files) else 'F5TTS_Base'
    
    test_phrase = 'Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!'
    ref_audio = '/content/F5-TTS/data/fefo/wavs/fefo_0001.wav'
    ref_text = 'fico feliz por você'
    out_dir = '/content/teste_fefo'
    os.makedirs(out_dir, exist_ok=True)
    
    !python src/f5_tts/infer/infer_cli.py \
        --model {model_name} \
        --ckpt_file "{best_ckpt}" \
        --ref_audio "{ref_audio}" \
        --ref_text "{ref_text}" \
        --gen_text "{test_phrase}" \
        --output_dir "{out_dir}"
    
    out_wav = glob.glob(f'{out_dir}/*.wav')
    if out_wav:
        print('=' * 60)
        print('✓ ÁUDIO GERADO COM A VOZ TREINADA DO FEFO! OUÇA ABAIXO:')
        display(ipd.Audio(out_wav[0]))
        print('=' * 60)
    
    # Exportar e baixar o modelo
    final_model_dest = '/content/fefo_model.pt' if best_ckpt.endswith('.pt') else '/content/fefo_model.safetensors'
    shutil.copyfile(best_ckpt, final_model_dest)
    print(f'🎉 TREINAMENTO CONCLUÍDO! Baixando {os.path.basename(final_model_dest)} para o seu PC...')
    files.download(final_model_dest)
else:
    print('Nenhum checkpoint encontrado.')
